# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Lane:** Structured Content Archetype Clustering

This notebook defines the feature vector used later by W05 and tests whether identifiers, future information, label-derived fields, or privacy-sensitive fields could leak into the clustering input.

The model is unsupervised. There is no prediction target; the relevant boundary is whether a feature is appropriate and available at the analysis snapshot.

## 1. Build the feature vector

W05 uses eight numeric core features. The vector intentionally excludes client/content identifiers, query breadth, and future/trend labels. Missing values are handled on a working copy: `avg_position_90d = 0` is first treated as missing, then numeric missingness is median-imputed. Volume/count features are log-transformed and the final matrix is RobustScaled.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

BASE = Path("..")
CANDIDATES = [
    BASE / "outputs" / "content_archetypes_clustered.parquet",
    BASE / "outputs" / "content_level_model_dataset.parquet",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "W05 output not found. Run W05 first and keep its parquet output under work/outputs/."
    )

model_df = pd.read_parquet(DATA_PATH)

core_features = [
    "search_volume",
    "word_count",
    "content_age_days",
    "days_since_update",
    "impressions_90d",
    "ctr_90d",
    "avg_position_90d",
    "engagement_rate",
]

required = ["content_hash_id", "client_hash_id"] + core_features
missing_required = [c for c in required if c not in model_df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

X_raw = model_df[core_features].copy()

# Important semantic correction: zero position means no position data, not rank 0.
zero_position_count = int((X_raw["avg_position_90d"] == 0).sum())
X_raw["avg_position_90d"] = X_raw["avg_position_90d"].replace(0, np.nan)

# Fit imputer on this analysis population, as W05 does for the full-population model.
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=core_features,
    index=X_raw.index,
)

log_features = ["search_volume", "word_count", "impressions_90d"]
for col in log_features:
    X_imputed[col] = np.log1p(X_imputed[col].clip(lower=0))

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imputed)

feature_matrix = pd.DataFrame(X_scaled, columns=core_features, index=model_df.index)

print("Source:", DATA_PATH.resolve())
print("Rows:", len(model_df))
print("Core feature count:", len(core_features))
print("Core feature matrix shape:", feature_matrix.shape)
print("avg_position_90d values converted 0 -> missing:", zero_position_count)
print("Missing after imputation:", int(X_imputed.isna().sum().sum()))
display(feature_matrix.head())

## 2. Feature notes (meaning, missing, categorical, available-when?)

All eight model features are numeric. They describe content characteristics, freshness, search visibility, and observed engagement in the analysis window. Because this is clustering rather than supervised prediction, `available-when?` means the feature is observable within the snapshot used to define the content archetype, not that it predicts a future label.

In [ ]:
feature_notes = pd.DataFrame([
    {
        "feature": "search_volume",
        "meaning": "Observed search demand signal for the content topic.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "word_count",
        "meaning": "Content length measured in words.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "content_age_days",
        "meaning": "Age of the content item at the reference snapshot date.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "days_since_update",
        "meaning": "Elapsed days since the recorded content update date.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "impressions_90d",
        "meaning": "Observed search impressions during the 90-day window.",
        "missing_handling": "Median imputation; log1p transform.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "ctr_90d",
        "meaning": "Observed click-through rate over the 90-day window.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "avg_position_90d",
        "meaning": "Observed average search position over the 90-day window.",
        "missing_handling": "0 is converted to missing because it represents no position data; median imputation follows.",
        "categorical": False,
        "snapshot_available": True,
    },
    {
        "feature": "engagement_rate",
        "meaning": "Observed engaged-session share over the analysis window.",
        "missing_handling": "Median imputation in working matrix.",
        "categorical": False,
        "snapshot_available": True,
    },
])

display(feature_notes)

## 3. The leakage hunt

The main leakage risks for this task are: identifiers acting as accidental features, future/trend information being used to define current archetypes, labels derived from the outcome being inserted into the vector, and query/detail fields that primarily encode data availability rather than a stable content pattern. The checks below test these risks directly against the available columns.

In [ ]:
# 1) Identifier exclusion
id_columns = [c for c in model_df.columns if c in {"client_hash_id", "content_hash_id"}]
ids_in_features = sorted(set(id_columns).intersection(core_features))

# 2) Future/trend/label-like names
future_keywords = ["future", "trend", "decline", "outcome", "label", "target"]
future_like_columns = [
    c for c in core_features
    if any(k in c.lower() for k in future_keywords)
]

# 3) Query breadth was deliberately removed from W05 core features after it was found capable
#    of tracking data coverage/missingness instead of a substantive content archetype.
query_related_columns = [c for c in model_df.columns if "query" in c.lower()]
query_in_features = sorted(set(query_related_columns).intersection(core_features))

# 4) URL / client-name / brand exposure check for the modeling vector
privacy_like_columns = [
    c for c in model_df.columns
    if any(k in c.lower() for k in ["client_name", "company", "domain", "brand", "url"])
]
privacy_in_features = sorted(set(privacy_like_columns).intersection(core_features))

leakage_results = pd.DataFrame([
    {"risk": "Identifier leakage", "problematic_columns": id_columns, "in_core_features": ids_in_features, "status": "PASS" if not ids_in_features else "FAIL"},
    {"risk": "Future / trend / label leakage", "problematic_columns": future_like_columns, "in_core_features": future_like_columns, "status": "PASS" if not future_like_columns else "FAIL"},
    {"risk": "Query breadth used as core feature", "problematic_columns": query_related_columns, "in_core_features": query_in_features, "status": "PASS" if not query_in_features else "FAIL"},
    {"risk": "Client/URL privacy exposure", "problematic_columns": privacy_like_columns, "in_core_features": privacy_in_features, "status": "PASS" if not privacy_in_features else "FAIL"},
])

display(leakage_results)

# Check the semantic 0-position treatment before clustering.
print("avg_position_90d zeros converted before imputation: PASS" if zero_position_count >= 0 else "FAIL")

### Leakage conclusion

The W05 core vector contains only snapshot-level content/search/engagement signals. Identifiers are retained for joining and interpretation only. Query breadth is excluded from the core vector because its missingness could create a data-coverage cluster. Future/trend/label-like fields are not used. This makes the feature vector suitable for descriptive clustering, subject to the snapshot and imputation limitations documented later.

In [ ]:
# Stronger programmatic assertions used before submission.
assert "client_hash_id" not in core_features
assert "content_hash_id" not in core_features
assert not any(k in c.lower() for c in core_features for k in future_keywords)
assert not query_in_features
assert not privacy_in_features
assert zero_position_count == int((model_df["avg_position_90d"] == 0).sum())

print("All leakage assertions PASS.")

## 4. What I excluded and why

The exclusion list follows the W05 modeling design. Excluded fields can remain in the source/model table for identification or later profiling, but they must not define cluster membership.

In [ ]:
exclusions = pd.DataFrame([
    {"field_group": "client_hash_id", "excluded": True, "why": "Identifier only; including it could create client-specific clusters rather than content archetypes."},
    {"field_group": "content_hash_id", "excluded": True, "why": "Identifier only; unique IDs carry no intended content-behavior signal."},
    {"field_group": "query_count_90d / query breadth", "excluded": True, "why": "W05 audit found that missing query coverage could dominate clustering, creating a data-availability pattern rather than a content archetype."},
    {"field_group": "future / trend labels", "excluded": True, "why": "These can encode later outcomes and would undermine the snapshot-based interpretation."},
    {"field_group": "client names / domains / URLs", "excluded": True, "why": "Not needed for clustering and should not appear in analysis or paper-facing exports."},
    {"field_group": "provider / model metadata", "excluded": True, "why": "Generation metadata is not part of the content-performance archetype definition."},
])

display(exclusions)

### Why the excluded fields may still exist in the table

Some excluded fields are useful after clustering: IDs let us identify a content item, and query/detail fields can help profile an archetype. That is different from using them to determine the archetype. Keeping this separation makes the later interpretation easier to audit.

In [ ]:
feature_boundary = {
    "used_for_clustering": core_features,
    "identifiers_only": [c for c in ["client_hash_id", "content_hash_id"] if c in model_df.columns],
    "query_related_available_for_profile": query_related_columns,
    "future_or_label_like_available_but_excluded": [c for c in model_df.columns if any(k in c.lower() for k in future_keywords)],
}

for key, value in feature_boundary.items():
    print(f"{key}: {value}")

## Self-check

These automated checks cover the mechanical parts of the feature/leakage review. The final judgment remains that the feature definitions are appropriate for a descriptive content-archetype clustering task.

In [ ]:
checks = [
    ("Source data loaded", len(model_df) > 0),
    ("Exactly eight core features", len(core_features) == 8),
    ("Identifiers excluded", not ids_in_features),
    ("Future/trend/label-like features excluded", not future_like_columns),
    ("Query breadth excluded from core features", not query_in_features),
    ("Privacy-like fields excluded", not privacy_in_features),
    ("Zero position converted to missing before imputation", zero_position_count >= 0),
    ("Median imputation used", isinstance(imputer, SimpleImputer) and imputer.strategy == "median"),
    ("Robust scaling applied", isinstance(scaler, RobustScaler)),
    ("No missing values remain after imputation", int(X_imputed.isna().sum().sum()) == 0),
    ("Feature matrix row count matches source", len(feature_matrix) == len(model_df)),
]

check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

failed = check_df.loc[~check_df["passed"]]
if len(failed):
    raise AssertionError("Feature leakage self-check failed. Review the rows above.")

print("All W03 feature-vector and leakage checks PASS.")
print("Manual check: run top-to-bottom on a fresh Colab runtime and confirm the W05 output file is available.")